In [ ]:
# !pip install -q ragas langchain-openai datasets tiktoken

## Métrica RAGAS

In [ ]:
# 70 centavos de dolar 15 minutos

In [ ]:
# ============================================
#  RAGAS LLM JUDGE → RLF (Faithfulness)
#  Uses OpenAI as LLM
#  Reads JSONL with predictions
# ============================================

import json
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness
from ragas.run_config import RunConfig
from langchain_openai import ChatOpenAI

# ==========================
# CONFIG
# ==========================
INPUT_JSONL = "responses-10 (1).jsonl"
OUTPUT_JSONL = "eval_results_ragas.jsonl"

OPENAI_API_KEY = "GPT API"
OPENAI_JUDGE_MODEL = "gpt-4o-mini"

# ==========================
# HELPERS
# ==========================
def extract_prediction(task):
    p = task.get("predictions", [])
    if isinstance(p, list) and len(p) > 0:
        return p[0].get("text", "")
    return ""

def extract_context_texts(task):
    """
    RAGAS espera: List[str] por ejemplo
    """
    ctxs = task.get("contexts", [])
    texts = []
    for c in ctxs:
        t = c.get("text", "")
        if t:
            texts.append(t)
    return texts

def extract_question(task):
    """
    RAGAS necesita una 'question', aunque
    no la usa directamente para faithfulness.
    Tomamos la última pregunta del input.
    """
    msgs = task.get("input", [])
    for m in reversed(msgs):
        if m.get("speaker") == "user":
            return m.get("text", "")
    return ""

# ==========================
# LOAD DATA
# ==========================
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    tasks = [json.loads(line) for line in f]

questions = []
answers = []
contexts = []

for t in tasks:
    questions.append(extract_question(t))
    answers.append(extract_prediction(t))
    contexts.append(extract_context_texts(t))

dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts,
})

# ==========================
# RAGAS EVALUATION
# ==========================
llm = ChatOpenAI(
    model=OPENAI_JUDGE_MODEL,
    api_key=OPENAI_API_KEY,
    temperature=0.0,
)

print(f"Running RAGAS Faithfulness on {len(dataset)} examples...\n")

result = evaluate(
    dataset,
    metrics=[faithfulness],
    llm=llm,
    run_config=RunConfig(timeout=120),
)

df_scores = result.to_pandas()

# ==========================
# ATTACH SCORES BACK
# ==========================
rlf_scores = df_scores["faithfulness"].values

for task, score in zip(tasks, rlf_scores):
    task.setdefault("metrics", {})
    task["metrics"]["RLF"] = float(score)

# ==========================
# RESULTS
# ==========================
print("\n====== RAGAS (RLF) RESULTS ======")
print(f"RLF (mean): {np.mean(rlf_scores):.4f}")

# ==========================
# SAVE
# ==========================
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for t in tasks:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")

print(f"\n✅ Saved: {OUTPUT_JSONL}")

Running RAGAS Faithfulness on 842 examples...



Evaluating:   0%|          | 0/842 [00:00<?, ?it/s]


====== RAGAS (RLF) RESULTS ======
RLF (mean): 0.7536

✅ Saved: eval_results_ragas.jsonl
